# 変分量子固有値ソルバー VQE
VQE (Variational Quantum Eigensolver) は、現代の量子コンピュータ上で実行される代表的な量子・古典ハイブリッドアルゴリズムであり、量子コンピュータと古典コンピュータの両方を利用します。

## 今回学ぶこと
1. VQEの基礎理論を学ぶ
2. Blueqatで変分量子回路を実装する

## 概要
理想的な誤り訂正機能を持つ量子コンピュータを前提とする汎用アルゴリズムに加えて、現在の量子コンピュータ(NISQ)向けの変分アルゴリズムがあります。

1. 汎用アルゴリズム(グローバー、ショア、位相推定、量子フーリエ変換、HHL、量子サポートベクターマシンなど)
2. 変分アルゴリズム(VQE, QAOA)

VQEは、短い量子回路を使って既存のコンピュータとハイブリッドに問題を解くアルゴリズムです。

## 固有値問題
VQEは、与えられた行列(ハミルトニアン)の固有値の期待値を計算します。固有値問題を解くことで様々な問題を解くことができます。固有値を $E_0$、固有ベクトルを $\mid \psi \rangle$ とすると、期待値は次のようになります。

$$
H \mid \psi \rangle = E_0 \mid \psi \rangle
$$

目的は $E_0$ を求めることです。

## ハミルトニアンと期待値

問題は、ハミルトニアン $H$ と呼ばれるエルミート行列として与えられます。ハミルトニアンはパウリ行列と単位行列から構成され、複素行列の形を取ります。

In [1]:
from blueqat.utils import X, Y, Z, I

h = 1.23 * I - 4.56 * X(0) + 2.45 * Y(0) + 2.34 * Z(0)
h.to_matrix()

tensor([[ 3.5700+0.0000j, -4.5600-2.4500j],
        [-4.5600+2.4500j, -1.1100+0.0000j]], dtype=torch.complex128)

ハミルトニアンの期待値は、

$$
\langle \psi \mid H \mid \psi \rangle
$$

であり、ハミルトニアンの期待値はユニタリ行列の線形結合であるため分解することができます。

$$
\langle \psi \mid aH_1 + bH_2 \mid \psi \rangle \\ = \langle \psi \mid aH_1 \mid \psi \rangle + \langle \psi \mid bH_2 \mid \psi \rangle \\ = a\langle \psi \mid H_1 \mid \psi \rangle + b\langle \psi \mid H_2 \mid \psi \rangle
$$

例えば、

$$
H = 1.2 X_0 Z_2 + 2.5 Z_0 X_1 Y_2 - 3.4 Z_2 X_1
$$

この式の期待値は以下のように求めることができます。

$$
\langle \psi \mid 1.2 X_0 Z_2 + 2.5 Z_0 X_1 Y_2 - 3.4 Z_2 X_1 \mid \psi \rangle\\
= 1.2\langle \psi \mid X_0 Z_2 \mid \psi \rangle + 2.5 \langle \psi \mid Z_0 X_1 Y_2\mid \psi \rangle - 3.4 \langle \psi \mid Z_2 X_1 \mid \psi \rangle
$$

## ハミルトニアンの期待値とサンプリング
ハミルトニアンの期待値は、計算結果のサンプリングから求めることができます。ハミルトニアン $H=Z$ に対する期待値は次のようになります。

$$
\langle \psi \mid Z \mid \psi \rangle = 
\begin{bmatrix}
\alpha^* & \beta^*
\end{bmatrix}
\begin{bmatrix}
1&0\\
0&-1
\end{bmatrix}
\begin{bmatrix}
\alpha\\
\beta
\end{bmatrix}
= |\alpha|^2 - |\beta|^2
$$

$|\alpha|^2$ と $|\beta|^2$ はそれぞれ0と1が得られる確率です。複数回計算を行い、そのサンプル値から期待値を求めます。

通常、ハミルトニアンがXまたはYの場合、サンプル値から直接期待値を求めることはできません。この場合、各軸の回転を利用してサンプルできるように調整します。

$X$ については $X = HZH$ を使います。

$$
\langle \psi \mid X \mid \psi \rangle \\
= \langle \psi \mid HZH \mid \psi \rangle\\
= \langle \psi' \mid Z \mid \psi' \rangle
$$

$Y$ については $Y = RX(-\pi/2) Z RX(\pi/2)$ を使います。

$$
\langle \psi \mid Y \mid \psi \rangle \\
= \langle \psi \mid RX(-\pi/2) Z RX(\pi/2) \mid \psi \rangle\\
= \langle \psi'' \mid Z \mid \psi'' \rangle
$$

この場合、測定の直前に対応する回転ゲートが挿入されます。

## 量子変分原理
任意の状態ベクトル $\psi(\theta)$ において、ハミルトニアンの期待値は次を満たします。

$$
\langle \psi (\theta) \mid H \mid \psi (\theta) \rangle \geq E_0
$$

VQEはこの量子変分原理を利用し、既存のコンピュータの最適化アルゴリズムを使って、角度パラメータ $\theta$ で状態ベクトルを変化させながら $E_0$ にできるだけ近い最小値を求めます。

## Ansatz
最小値を効率よく求めるための量子回路はAnsatz(アンザッツ)と呼ばれます。現在では、量子化学向けのUCC Ansatzや組合せ最適化問題向けのQAOA Ansatzなど、効率的な量子回路が発見されており応用が期待されています。Ansatzは分野ごとにある程度の作法があり、その作法に従って記述されます。

## 例
最後に、実際に例を試してみましょう。

1. 回転ゲートの角度をパラメータとするAnsatzを作成する。(量子)
2. 実行結果から $\langle \psi (\theta) \mid H \mid \psi (\theta) \rangle$ を計算する。(古典)
3. 古典オプティマイザから次の角度パラメータの候補を試す。

今回はAnsatzとして任意のものを使います。

```
rx(a)[0].rz(b)[0]
```

aとbという2つの角度を使って1量子ビットを計算します。ハミルトニアンには上で出てきた例を使います。最後に、VQEの計算結果を数値計算ライブラリnumpyの結果と比較してみましょう。

In [2]:
import numpy as np
import scipy.optimize as optimize

from blueqat import Circuit
from blueqat.utils import X, Y, Z, I
    
# hamiltonian
hamiltonian = 1.23 * I - 4.56 * X(0) + 2.45 * Y(0) + 2.34 * Z(0)
hamiltonian = hamiltonian.to_expr()

def f(params):
    params = params
    return Circuit().rx(params[0])[0].rz(params[1])[0].run(hamiltonian = hamiltonian)

initial_guess = np.array([np.random.rand()*np.pi*2 for _ in range(2)])
result = optimize.minimize(f, initial_guess, method="Powell", options={"ftol": 5.0e-2, "xtol": 5.0e-2, "maxiter": 1000})

print('Result by VQE')
print(Circuit().rx(result.x[0])[0].rz(result.x[1])[0].run(hamiltonian = hamiltonian))

# This is for check
mat = h.to_matrix().numpy()
print('Result by numpy')
print(np.linalg.eigh(mat)[0][0])

Result by VQE
tensor(-4.2683, dtype=torch.float64)
Result by numpy
-4.4508186029832
